In [1]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image, make_grid
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
data_dir = Path(r"C:\Users\User\Desktop\Data")

images_path = data_dir / "Images.npy"
metadata_path = data_dir / "metadata_encoded.npy"

save_dir = data_dir / "diffusion_256_conditional"
save_dir.mkdir(exist_ok=True)

model_path = save_dir / "model_256_conditional.pth"

In [3]:
class NpyImageConditionDataset(Dataset):
    def __init__(self, images_path, metadata_path):
        self.images = np.load(images_path)
        self.metadata = np.load(metadata_path)

        if self.images.ndim != 2 or self.images.shape[1] != 256 * 256:
            raise ValueError(f"Images.npy must have shape (N, 65536), got {self.images.shape}")

        if len(self.images) != len(self.metadata):
            raise ValueError(
                f"Images and metadata row count mismatch: "
                f"{len(self.images)} vs {len(self.metadata)}"
            )

        self.images = self.images.astype(np.float32) / 255.0
        self.images = self.images * 2.0 - 1.0

        self.metadata = self.metadata.astype(np.float32)

        print("Images shape:", self.images.shape)
        print("Metadata shape:", self.metadata.shape)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        image = image.view(1, 256, 256)

        condition = torch.tensor(self.metadata[idx], dtype=torch.float32)

        return image, condition

In [4]:
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, is_res=False):
        super().__init__()

        self.is_res = is_res
        self.same_channels = in_channels == out_channels

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

        if is_res and not self.same_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x1)

        if self.is_res:
            return (self.shortcut(x) + x2) / 1.414

        return x2


class UnetDown(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.model = nn.Sequential(
            ResidualConvBlock(in_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.model(x)


class UnetUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.model = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        )

    def forward(self, x, skip):
        x = torch.cat((x, skip), dim=1)
        return self.model(x)


class EmbedFC(nn.Module):
    def __init__(self, input_dim, emb_dim):
        super().__init__()

        self.input_dim = input_dim

        self.model = nn.Sequential(
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, x):
        x = x.view(-1, self.input_dim)
        return self.model(x)

In [5]:
class ContextUnet(nn.Module):
    def __init__(self, in_channels, n_feat, n_cfeat, height):
        super().__init__()

        self.in_channels = in_channels
        self.n_feat = n_feat
        self.n_cfeat = n_cfeat
        self.h = height

        self.init_conv = ResidualConvBlock(in_channels, n_feat, is_res=True)

        self.down1 = UnetDown(n_feat, n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)

        self.to_vec = nn.Sequential(
            nn.AvgPool2d(self.h // 4),
            nn.GELU(),
        )

        self.timeembed1 = EmbedFC(1, 2 * n_feat)
        self.timeembed2 = EmbedFC(1, n_feat)

        self.contextembed1 = EmbedFC(n_cfeat, 2 * n_feat)
        self.contextembed2 = EmbedFC(n_cfeat, n_feat)

        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, self.h // 4, self.h // 4),
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )

        self.up1 = UnetUp(4 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)

        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, in_channels, 3, 1, 1),
        )

    def forward(self, x, t, c):
        x = self.init_conv(x)

        down1 = self.down1(x)
        down2 = self.down2(down1)

        hiddenvec = self.to_vec(down2)

        cemb1 = self.contextembed1(c).view(-1, self.n_feat * 2, 1, 1)
        temb1 = self.timeembed1(t).view(-1, self.n_feat * 2, 1, 1)

        cemb2 = self.contextembed2(c).view(-1, self.n_feat, 1, 1)
        temb2 = self.timeembed2(t).view(-1, self.n_feat, 1, 1)

        up1 = self.up0(hiddenvec)
        up2 = self.up1(cemb1 * up1 + temb1, down2)
        up3 = self.up2(cemb2 * up2 + temb2, down1)

        out = self.out(torch.cat((up3, x), dim=1))

        return out

In [6]:
timesteps = 500
beta1 = 1e-4
beta2 = 0.02

height = 256
in_channels = 1

n_feat = 16
batch_size = 2
n_epoch = 100
lrate = 1e-4

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda:0


In [7]:
dataset = NpyImageConditionDataset(images_path, metadata_path)

n_cfeat = dataset.metadata.shape[1]

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

print("Condition vector size:", n_cfeat)

Images shape: (968, 65536)
Metadata shape: (968, 6)
Condition vector size: 6


In [8]:
nn_model = ContextUnet(
    in_channels=in_channels,
    n_feat=n_feat,
    n_cfeat=n_cfeat,
    height=height,
).to(device)

optim = torch.optim.Adam(nn_model.parameters(), lr=lrate)

print("Model created")

Model created


In [9]:
b_t = (beta2 - beta1) * torch.linspace(0, 1, timesteps + 1, device=device) + beta1
a_t = 1 - b_t
ab_t = torch.cumsum(a_t.log(), dim=0).exp()
ab_t[0] = 1

In [10]:
def perturb_input(x, t, noise):
    return (
        ab_t.sqrt()[t, None, None, None] * x
        + (1 - ab_t[t, None, None, None]).sqrt() * noise
    )

In [11]:
nn_model.train()

for ep in range(n_epoch):
    print(f"epoch {ep + 1}/{n_epoch}")

    optim.param_groups[0]["lr"] = lrate * (1 - ep / n_epoch)

    pbar = tqdm(dataloader, mininterval=2)

    for x, c in pbar:
        x = x.to(device)
        c = c.to(device)

        optim.zero_grad()

        noise = torch.randn_like(x)
        t = torch.randint(1, timesteps + 1, (x.shape[0],), device=device)

        x_pert = perturb_input(x, t, noise)

        pred_noise = nn_model(
            x_pert,
            t.float() / timesteps,
            c,
        )

        loss = F.mse_loss(pred_noise, noise)

        loss.backward()
        optim.step()

        pbar.set_description(f"loss: {loss.item():.5f}")

torch.save(nn_model.state_dict(), model_path)

print("Saved model:", model_path)

epoch 1/100


loss: 0.05229: 100%|██████████| 484/484 [00:21<00:00, 22.65it/s]


epoch 2/100


loss: 0.02512: 100%|██████████| 484/484 [00:20<00:00, 23.99it/s]


epoch 3/100


...

epoch 99/100


loss: 0.00222: 100%|██████████| 484/484 [00:20<00:00, 23.34it/s]


epoch 100/100


loss: 0.00436: 100%|██████████| 484/484 [00:20<00:00, 23.30it/s]


Saved model: C:\Users\User\Desktop\Data\diffusion_256_conditional\model_256_conditional.pth


In [12]:
def denoise_add_noise(x, t, pred_noise, z=None):
    if z is None:
        z = torch.randn_like(x)

    noise = b_t.sqrt()[t] * z

    mean = (
        x - pred_noise * ((1 - a_t[t]) / (1 - ab_t[t]).sqrt())
    ) / a_t[t].sqrt()

    return mean + noise

In [13]:
@torch.no_grad()
def sample_ddpm(condition_vectors, save_rate=20):
    nn_model.eval()

    condition_vectors = condition_vectors.to(device)
    n_sample = condition_vectors.shape[0]

    samples = torch.randn(n_sample, 1, height, height, device=device)

    intermediate = []

    for i in range(timesteps, 0, -1):
        print(f"sampling timestep {i:3d}", end="\r")

        t = torch.full(
            (n_sample,),
            i / timesteps,
            device=device,
            dtype=torch.float32,
        )

        z = torch.randn_like(samples) if i > 1 else 0

        eps = nn_model(samples, t, condition_vectors)
        samples = denoise_add_noise(samples, i, eps, z)

        if i % save_rate == 0 or i == timesteps or i < 8:
            intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)

    return samples, intermediate

In [14]:
nn_model.load_state_dict(torch.load(model_path, map_location=device))
nn_model.eval()

print("Loaded model:", model_path)

Loaded model: C:\Users\User\Desktop\Data\diffusion_256_conditional\model_256_conditional.pth


In [21]:
metadata = np.load(metadata_path).astype(np.float32)

condition_vectors = torch.tensor(metadata[10:14], dtype=torch.float32)

samples, intermediate = sample_ddpm(condition_vectors)

samples = samples.clamp(-1, 1)
samples = (samples + 1) / 2

output_image_path = save_dir / "sample_output.png"

save_image(samples, output_image_path, nrow=1)

print("Saved sample image:", output_image_path)

Saved sample image: C:\Users\User\Desktop\Data\diffusion_256_conditional\sample_output.png
